In [0]:
%pip install scikit-learn azure-storage-blob

In [0]:
from pyspark.sql import functions as F
import datetime

# ── Configuration ─────────────────────────────────────────────────────────────
STORAGE_ACCOUNT = "stockpipelinedl"
STORAGE_KEY = dbutils.secrets.get(scope="stock-pipeline", key="azure-storage-key")
CONTAINER_GOLD = "gold"

# ── Dynamic date ──────────────────────────────────────────────────────────────
try:
    DATE = dbutils.widgets.get("execution_date")
except:
    # DATE = "2026-05-20"
    DATE = (datetime.datetime.utcnow() - datetime.timedelta(days=1)).strftime('%Y-%m-%d')

print(f"Processing date: {DATE}")

# ── Connect Spark to Azure ────────────────────────────────────────────────────
spark.conf.set(
    f"fs.azure.account.key.{STORAGE_ACCOUNT}.dfs.core.windows.net",
    STORAGE_KEY
)

print("✅ Configuration done")

In [0]:
import pandas as pd

# ── Read Gold Delta table ─────────────────────────────────────────────────────
gold_path = f"abfss://{CONTAINER_GOLD}@{STORAGE_ACCOUNT}.dfs.core.windows.net/stock_data/"

gold_df = spark.read.format("delta").load(gold_path)

print(f"✅ Read {gold_df.count()} records from Gold")

# ── Convert to Pandas for ML model ───────────────────────────────────────────
gold_pandas = gold_df.toPandas()

print(f"\nColumns available: {list(gold_pandas.columns)}")
from IPython.display import display
display(gold_pandas)

In [0]:
# ── Debug: Check Gold data ────────────────────────────────────────────────────
print(f"Total records in Gold: {len(gold_pandas)}")
print(f"\nColumns: {list(gold_pandas.columns)}")
print(f"\nNull counts per column:")
print(gold_pandas.isnull().sum())
print(f"\nSample data:")
display(gold_pandas.head(20))

In [0]:
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
import numpy as np

# ── Select features for anomaly detection ────────────────────────────────────
# These are the features the model will use to detect unusual patterns
features = [
    'close_price',
    'daily_return_pct',
    'price_volatility',
    'price_range_pct',
    'volume'
]

# ── Handle nulls — drop rows where features are null ─────────────────────────
model_df = gold_pandas[features].dropna()
valid_indices = model_df.index

print(f"Records used for training: {len(model_df)}")
print(f"\nFeature statistics:")
display(model_df.describe())

# ── Scale features — important for ML models ──────────────────────────────────
# StandardScaler converts all features to same scale
# so volume (millions) doesn't dominate close_price (hundreds)
scaler = StandardScaler()
if not model_df.empty:    scaled_features = scaler.fit_transform(model_df)
else:
    scaled_features = np.array([])


# ── Train Isolation Forest ────────────────────────────────────────────────────
# contamination=0.1 means we expect ~10% of data points to be anomalies
model = IsolationForest(
    contamination=0.1,
    random_state=42,
    n_estimators=100
)

predictions = model.fit_predict(scaled_features)
anomaly_scores = model.score_samples(scaled_features)

# Isolation Forest returns: -1 = anomaly, 1 = normal
# We convert to: True = anomaly, False = normal
gold_pandas.loc[valid_indices, 'is_anomaly'] = predictions == -1
gold_pandas.loc[valid_indices, 'anomaly_score'] = anomaly_scores

# Fill nulls for rows that were dropped
gold_pandas['is_anomaly'] = gold_pandas['is_anomaly'].fillna(False)
gold_pandas['anomaly_score'] = gold_pandas['anomaly_score'].fillna(0)

print(f"\n✅ Model trained!")
print(f"   Total records: {len(gold_pandas)}")
print(f"   Anomalies detected: {gold_pandas['is_anomaly'].sum()}")
print(f"   Normal records: {(~gold_pandas['is_anomaly']).sum()}")

In [0]:
# ── Anomaly Report ────────────────────────────────────────────────────────────
print("=" * 60)
print("ANOMALY DETECTION REPORT")
print("=" * 60)

anomalies = gold_pandas[gold_pandas['is_anomaly'] == True]
normal = gold_pandas[gold_pandas['is_anomaly'] == False]

if len(anomalies) > 0:
    print(f"\n⚠️ ANOMALIES DETECTED ({len(anomalies)} records):")
    display(anomalies[[
        'ticker', 
        'trade_date', 
        'close_price', 
        'daily_return_pct',
        'price_volatility',
        'volume',
        'anomaly_score'
    ]])
else:
    print("\n✅ No anomalies detected!")

print(f"\n✅ NORMAL RECORDS ({len(normal)} records):")
display(normal[[
    'ticker',
    'trade_date', 
    'close_price',
    'daily_return_pct',
    'anomaly_score'
]])

In [0]:
# ── Convert back to Spark DataFrame ──────────────────────────────────────────
anomaly_spark_df = spark.createDataFrame(gold_pandas)

# ── Write anomaly results to Gold container ───────────────────────────────────
anomaly_path = f"abfss://{CONTAINER_GOLD}@{STORAGE_ACCOUNT}.dfs.core.windows.net/anomaly_report/"

anomaly_spark_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .save(anomaly_path)

print(f"✅ Anomaly report written to Gold container!")
print(f"   Path: {anomaly_path}")
print(f"   Total records: {anomaly_spark_df.count()}")
print(f"   Anomalies: {gold_pandas['is_anomaly'].sum()}")

# ── Verify ────────────────────────────────────────────────────────────────────
verify_df = spark.read.format("delta").load(anomaly_path)
print(f"\n✅ Verification - Records in Anomaly Report: {verify_df.count()}")
display(verify_df.toPandas())